In [ ]:
!pip install -q scikit-learn openpyxl transformers

import pandas as pd
import re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
from scipy.sparse import hstack
import os
import joblib

# =====================================================
# Load Dataset
# =====================================================
train_df = pd.read_excel('/content/MOLDV2_train.csv.xlsx')
test_df = pd.read_excel('/content/MOLDV2_test.csv.xlsx')

# =====================================================
# Clean Marathi text
# =====================================================
def clean_marathi_text(text):
    text = str(text)
    text = re.sub(r"http\S+|@\w+|#\w+|\d+", "", text)  # remove links, hashtags, digits
    text = re.sub(r"[^\w\s\u0900-\u097F]", "", text)  # keep only Devanagari + alphanumeric
    return re.sub(r"\s+", " ", text).strip()

# =====================================================
# L3Cube MahaHate-BERT Feature Extractor
# =====================================================
class L3CubeFeatureExtractor:
    def __init__(self, model_name="l3cube-pune/mahahate-bert"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def extract_features(self, texts, max_length=128, batch_size=16):
        features = []
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]
                inputs = self.tokenizer(
                    batch_texts,
                    max_length=max_length,
                    padding='max_length',
                    truncation=True,
                    return_tensors='pt'
                )
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                outputs = self.model(**inputs)
                cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                features.extend(cls_embeddings)
        return np.array(features)

# =====================================================
# TF-IDF Vectorizers
# =====================================================
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=3000)
char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=3000)
combined_vectorizer = FeatureUnion([('word', word_vectorizer), ('char', char_vectorizer)])

# =====================================================
# Function to train + evaluate each subtask (with tuning)
# =====================================================
def run_subtask(task_col):
    print(f"\n--- {task_col} ---")

    train_data = train_df.dropna(subset=['tweet', task_col]).copy()
    test_data = test_df.dropna(subset=['tweet', task_col]).copy()

    train_data['clean_tweet'] = train_data['tweet'].apply(clean_marathi_text)
    test_data['clean_tweet'] = test_data['tweet'].apply(clean_marathi_text)

    label_enc = LabelEncoder()
    y_train = label_enc.fit_transform(train_data[task_col])
    y_test = label_enc.transform(test_data[task_col])

    X_train_tfidf = combined_vectorizer.fit_transform(train_data['clean_tweet'])
    X_test_tfidf = combined_vectorizer.transform(test_data['clean_tweet'])

    bert_extractor = L3CubeFeatureExtractor()
    X_train_bert = bert_extractor.extract_features(train_data['clean_tweet'].astype(str).tolist())
    X_test_bert = bert_extractor.extract_features(test_data['clean_tweet'].astype(str).tolist())

    X_train_combined = hstack([X_train_tfidf, X_train_bert])
    X_test_combined = hstack([X_test_tfidf, X_test_bert])

    param_grid = {
        'n_estimators': [100, 300],
        'max_depth': [10, 30, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
        'max_features': ['sqrt', 'log2']
    }

    grid_rf = GridSearchCV(
        estimator=RandomForestClassifier(random_state=42),
        param_grid=param_grid,
        scoring='f1_macro',
        cv=3,
        n_jobs=-1,
        verbose=1
    )
    grid_rf.fit(X_train_combined, y_train)
    model = grid_rf.best_estimator_

    print("\nBest Parameters:", grid_rf.best_params_)

    preds = model.predict(X_test_combined)
    print("Classification Report:")
    print(classification_report(y_test, preds, target_names=label_enc.classes_))
    macro_f1 = f1_score(y_test, preds, average='macro')
    acc = accuracy_score(y_test, preds)
    print(f"Accuracy: {acc:.4f}, Macro F1: {macro_f1:.4f}")

    # Save model + label encoder
    save_path = '/content/drive/MyDrive/mold_models'
    os.makedirs(save_path, exist_ok=True)
    joblib.dump(model, f"{save_path}/random_forest_{task_col}.joblib")
    joblib.dump(label_enc, f"{save_path}/label_encoder_{task_col}.joblib")

    print(f"Model and encoder saved for {task_col}.")

In [ ]:
run_subtask('subtask_a')
run_subtask('subtask_b')
run_subtask('subtask_c')



--- subtask_a ---


tokenizer_config.json:   0%|          | 0.00/305 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Fitting 3 folds for each of 48 candidates, totalling 144 fits

Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Classification Report:
               precision    recall  f1-score   support

    Offensive       0.95      1.00      0.97       260
not offensive       1.00      0.94      0.97       250

     accuracy                           0.97       510
    macro avg       0.97      0.97      0.97       510
 weighted avg       0.97      0.97      0.97       510

Accuracy: 0.9686, Macro F1: 0.9686
Model and encoder saved for subtask_a.

--- subtask_b ---


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Fitting 3 folds for each of 48 candidates, totalling 144 fits

Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Classification Report:
              precision    recall  f1-score   support

         TIN       0.98      0.99      0.99       158
         UNT       0.99      0.97      0.98       102

    accuracy                           0.98       260
   macro avg       0.99      0.98      0.98       260
weighted avg       0.98      0.98      0.98       260

Accuracy: 0.9846, Macro F1: 0.9838
Model and encoder saved for subtask_b.

--- subtask_c ---


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Fitting 3 folds for each of 48 candidates, totalling 144 fits

Best Parameters: {'max_depth': 30, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Classification Report:
              precision    recall  f1-score   support

         GRP       1.00      1.00      1.00        51
         IND       0.98      1.00      0.99        51
         OTH       1.00      0.98      0.99        56

    accuracy                           0.99       158
   macro avg       0.99      0.99      0.99       158
weighted avg       0.99      0.99      0.99       158

Accuracy: 0.9937, Macro F1: 0.9938
Model and encoder saved for subtask_c.


In [ ]:
!ls /content


drive  MOLDV2_test.csv.xlsx  MOLDV2_train.csv.xlsx  sample_data
